In [ ]:
!pip install torchmetrics

In [ ]:
import torch
import torchvision
from pathlib import Path
from torchmetrics.detection import MeanAveragePrecision
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/datasets/player_football/set1"
NUM_CLASSES = 3
NUM_EPOCHS = 20
BATCH_SIZE = 4
LR = 0.005

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

In [ ]:
class YOLOLabelDataset(Dataset):
    def __init__(self, root, split, transforms=None):
        self.img_dir = Path(root) / split / "images"
        self.lbl_dir = Path(root) / split / "labels"
        self.transforms = transforms
        self.images = sorted(self.img_dir.iterdir())

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_filepath = self.images[idx]
        base_filename = img_filepath.stem

        image = Image.open(img_filepath).convert("RGB")
        if self.transforms:
            img_tensor = self.transforms(image)

        bbox_coords = []
        labels = []

        img_w, img_h = image.size

        lbl_filepath = self.lbl_dir / (base_filename + ".txt")
        lines = Path(lbl_filepath).read_text().splitlines()
        
        for line in lines:
            parts = line.split()
            labels.append(int(parts[0]) + 1)
            bbox_xyxy = self.yolo_to_xyxy(parts[1:], img_w, img_h)
            bbox_coords.append(bbox_xyxy)
        target = {
            "boxes": torch.tensor(bbox_coords, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long)
        }
        return (img_tensor, target)

    def yolo_to_xyxy(self, bbox, img_w, img_h):
        cx, cy, bw, bh = [float(x) for x in bbox]
        x1 = (cx - bw/2) * img_w
        y1 = (cy - bh/2) * img_h
        x2 = (cx + bw/2) * img_w
        y2 = (cy + bh/2) * img_h

        return [x1, y1, x2, y2]